# Compliance Copilot - October milestone (Ollama, fully local)
Non-agentic RAG baseline: Ollama embeddings (`nomic-embed-text`) → vector search over NPR 7150.2D clauses →
Ollama `qwen3:4b` judge forced into a Pydantic schema → grounding check → precision/recall vs. the human-labeled benchmark.

**Before running:** Runtime → Change runtime type → **T4 GPU**. (CPU works but is very slow for the full 50 rows.)
Run cells top to bottom. The upload cell (step 3) can be re-run any time to add data.

## 1. Install Ollama (one-time per runtime)

In [ ]:
%%bash
set -euo pipefail
if ! command -v zstd >/dev/null 2>&1; then
  apt-get update --quiet
  DEBIAN_FRONTEND=noninteractive apt-get install -y zstd
fi
if ! command -v ollama >/dev/null 2>&1 || \
   ! ollama --version 2>&1 | grep -Eq '(^|[^0-9.])0\.33\.3([^0-9.]|$)'; then
  curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.33.3 sh
fi
python -m pip install --quiet ollama==0.6.2 pydantic==2.13.5 tabulate faiss-cpu
ollama --version

## 2. Start the server and pull the models

In [ ]:
import os, subprocess, time, requests
from pathlib import Path

GENERATION_MODEL = "qwen3:4b"   # optional upgrade on a T4: "qwen3:8b" (re-run this cell to pull it)
EMBEDDING_MODEL  = "nomic-embed-text"
HOST = "http://127.0.0.1:11434"
os.environ["OLLAMA_HOST"] = HOST          # the script reads this

def ready():
    try: return requests.get(f"{HOST}/api/tags", timeout=2).ok
    except requests.RequestException: return False

if not ready():
    log = open("/tmp/ollama-colab.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
t_end = time.monotonic() + 60
while not ready() and time.monotonic() < t_end:
    time.sleep(1)
assert ready(), "Ollama did not start:\n" + Path("/tmp/ollama-colab.log").read_text()[-3000:]
print("Ollama server ready.")

for m in (EMBEDDING_MODEL, GENERATION_MODEL):
    print("pulling", m); subprocess.run(["ollama", "pull", m], check=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU detected - this will be slow"

## 3. Upload code + data
Select `compliance_rag_baseline.py` and your CSVs (clause corpus + benchmark) together.
**Re-run this cell later to add more files** - nothing already uploaded is deleted.
Prefer Drive? Set `USE_DRIVE = True` and point `DRIVE_DIR` at a folder with the files.

In [ ]:
import os, shutil, glob
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/compliance_copilot"   # only used if USE_DRIVE

DATA = "/content/data"; os.makedirs(DATA, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for f in glob.glob(f"{DRIVE_DIR}/*"):
        shutil.copy(f, "/content" if f.endswith(".py") else DATA)
else:
    from google.colab import files
    for name, content in files.upload().items():
        dest = "/content" if name.endswith(".py") else DATA
        with open(f"{dest}/{name}", "wb") as fh:
            fh.write(content)

# Colab renames re-uploads ("name (2).py"): copy the newest matching script to the importable name
cands = glob.glob("/content/*rag*baseline*.py")
if cands:
    newest = max(cands, key=os.path.getmtime)
    if os.path.abspath(newest) != "/content/compliance_rag_baseline.py":
        shutil.copy(newest, "/content/compliance_rag_baseline.py")
print("data files:", sorted(os.listdir(DATA)))
print("script present:", os.path.exists("/content/compliance_rag_baseline.py"))

## 4. Auto-detect files by their columns
- **Clause corpus**: has `clause` + `requirement` columns
- **Benchmark**: has `requirement_text`, `clause_id`, `label`. Multiple benchmark files are combined (duplicates dropped).

In [ ]:
import sys, glob, pandas as pd, importlib
sys.path.insert(0, "/content")
import compliance_rag_baseline as crb
importlib.reload(crb)

clause_files, bench_files = [], []
for f in sorted(glob.glob(f"{DATA}/*.csv")):
    cols = set(pd.read_csv(f, nrows=0).columns)
    if {"clause", "requirement"} <= cols: clause_files.append(f)
    elif {"requirement_text", "clause_id", "label"} <= cols: bench_files.append(f)
    else: print("skipped (unrecognised columns):", f)

assert clause_files, "No clause corpus CSV found - upload the NPR 7150.2D requirements CSV"
assert bench_files, "No benchmark CSV found - upload the ground-truth benchmark"

clauses_csv = max(clause_files, key=os.path.getmtime)
bench = pd.concat([pd.read_csv(f, dtype=str) for f in bench_files], ignore_index=True)
before = len(bench)
bench = bench.drop_duplicates(subset=["requirement_text", "clause_id"]).reset_index(drop=True)
bench["requirement_id"] = range(1, len(bench) + 1)
COMBINED = "/content/data_combined_benchmark.csv"
bench.to_csv(COMBINED, index=False)

print("clauses:", clauses_csv)
print("benchmark files:", [os.path.basename(f) for f in bench_files])
print(f"benchmark rows: {len(bench)} ({before - len(bench)} duplicates dropped)")
print(bench["label"].value_counts().to_string())

## 5. Run the baseline
Dry run first with `LIMIT = 5`, then `LIMIT = None` for all rows. Finished rows are cached, so re-running resumes.

In [ ]:
RUN_NAME = "v2"    # change per experiment so results never overwrite each other (e.g. "v1_qwen4b", "v2_qwen8b")
LIMIT = 5          # set to None for the full run
K = 5              # clauses retrieved per requirement
# Cache survives a runtime restart only if it lives on Drive, e.g.
OUT = f"/content/out_{RUN_NAME}"
# CACHE = f"/content/drive/MyDrive/compliance_copilot/cache_{RUN_NAME}.jsonl"
CACHE = f"{OUT}/cache.jsonl"

args = ["--clauses", clauses_csv, "--benchmark", COMBINED,
        "--embedder", "ollama", "--ollama-embed-model", EMBEDDING_MODEL,
        "--judge", "ollama", "--model", GENERATION_MODEL,
        "--k", str(K), "--out", OUT, "--cache", CACHE]
if LIMIT: args += ["--limit", str(LIMIT)]
df, metrics = crb.main(args)

## 6. Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/report.md").read()))

In [ ]:
df[["requirement_id","requirement_text","label","pred_verdict","clause_id","pred_clause","pred_rationale"]].head(20)

## 7. Download outputs

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"/content/october_results_{RUN_NAME}", "zip", OUT)
files.download(f"/content/october_results_{RUN_NAME}.zip")